# Pandas vs SQL Cheat Sheet (Terry Pratchett Discworld Edition)

This notebook provides a side-by-side comparison of common operations in pandas and SQL (SQLite3), using a fun dataset based on Terry Pratchett's Discworld books.

**Key Fixes in this version:**
* Ensured SQL tables and Pandas DataFrames stay in sync during updates.
* Fixed the broken code at the end of the original notebook.
* Added proper imports and display settings.

In [ ]:
import pandas as pd
import numpy as np
import sqlite3

# Set pandas options for better display
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Create SQLite connection
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

## 1. Creating Data Structures
We initialize both the Pandas DataFrames and the SQL Tables with the same data.

In [ ]:
print("--- 1. SETUP & DATA CREATION ---")

# Pandas: Create DataFrames
books_data = {
    'book_id': [1, 2, 3, 4, 5],
    'title': ['The Colour of Magic', 'Guards! Guards!', 'Mort', 'Going Postal', 'Small Gods'],
    'pub_year': [1983, 1989, 1987, 2004, 1992],
    'pages': [288, 355, 304, 394, 400],
    'main_character': ['Rincewind', 'Sam Vimes', 'Death', 'Moist von Lipwig', 'Brutha'],
    'sub_series': ['Rincewind', 'City Watch', 'Death', 'Moist von Lipwig', 'One-off']
}
books_df = pd.DataFrame(books_data)

characters_data = {
    'char_id': [1, 2, 3, 4, 5, 6],
    'name': ['Rincewind', 'Sam Vimes', 'Death', 'Moist von Lipwig', 'Brutha', 'Granny Weatherwax'],
    'occupation': ['Wizzard', 'Guard', 'Anthropomorphic Personification', 'Postmaster', 'Novice', 'Witch'],
    'location': ['Unseen University', 'Ankh-Morpork', "Death's Domain", 'Ankh-Morpork', 'Omnia', 'Lancre'],
    'first_appearance': [1, 8, 1, 33, 13, 6]
}
characters_df = pd.DataFrame(characters_data)

# SQL: Create Tables and Insert Data
# We use to_sql for cleaner setup, mimicking the INSERT statements
books_df.to_sql('books', conn, if_exists='replace', index=False)
characters_df.to_sql('characters', conn, if_exists='replace', index=False)

print("Data loaded successfully.")

## 2. Adding Columns
We add an 'author' column to both.

In [ ]:
print("\n--- 2. ADDING COLUMNS ---")

# Pandas: Add 'author' column
books_df['author'] = 'Terry Pratchett'

# SQL: Add 'author' column
cursor.execute("ALTER TABLE books ADD COLUMN author TEXT DEFAULT 'Terry Pratchett'")
# (Note: SQLite ADD COLUMN sets NULL for existing rows unless DEFAULT is specified, 
# but pandas to_sql above doesn't support defaults easily, so we update manually for consistency)
cursor.execute("UPDATE books SET author = 'Terry Pratchett'")
conn.commit()

print(books_df[['title', 'author']].head(2))

## 3. Adding Rows
Adding 'Wyrd Sisters' to the dataset.

In [ ]:
print("\n--- 3. ADDING ROWS ---")

# Pandas: Add a new book
new_book = pd.DataFrame([{
    'book_id': 6,
    'title': 'Wyrd Sisters',
    'pub_year': 1988,
    'pages': 265,
    'main_character': 'Granny Weatherwax',
    'sub_series': 'Witches',
    'author': 'Terry Pratchett'
}])
books_df = pd.concat([books_df, new_book], ignore_index=True)

# SQL: Add a new book
cursor.execute("""
INSERT INTO books (book_id, title, pub_year, pages, main_character, sub_series, author)
VALUES (6, 'Wyrd Sisters', 1988, 265, 'Granny Weatherwax', 'Witches', 'Terry Pratchett');
""")
conn.commit()

print(books_df.tail(2))

## 4. Updating Values
Changing 'One-off' series to 'Standalone'.

In [ ]:
print("\n--- 4. UPDATING VALUES ---")

# Pandas: Update 'One-off' to 'Standalone'
books_df.loc[books_df['sub_series'] == 'One-off', 'sub_series'] = 'Standalone'

# SQL: Update 'One-off' to 'Standalone'
cursor.execute("UPDATE books SET sub_series = 'Standalone' WHERE sub_series = 'One-off'")
conn.commit()

# Verify update
print(books_df[books_df['title'] == 'Small Gods'][['title', 'sub_series']])

## 5. Aggregations
Calculating average pages per series.

In [ ]:
print("\n--- 5. AGGREGATIONS (Avg Pages per Series) ---")

# Pandas
print("Pandas Result:")
print(books_df.groupby('sub_series')['pages'].mean())

# SQL
print("\nSQL Result:")
print(pd.read_sql("SELECT sub_series, AVG(pages) as avg_pages FROM books GROUP BY sub_series", conn))

## 6. Joining Data
Performing a Left Join between Books and Characters.

In [ ]:
print("\n--- 6. LEFT JOIN ---")

# Pandas Left Join
pandas_join = pd.merge(
    books_df,
    characters_df,
    left_on='main_character',
    right_on='name',
    how='left'
)
print("Pandas Left Join (First 3 cols):")
print(pandas_join[['title', 'main_character', 'occupation']].tail())

# SQL Left Join
sql_join = pd.read_sql("""
SELECT b.title, b.main_character, c.occupation
FROM books b
LEFT JOIN characters c ON b.main_character = c.name
""", conn)

print("\nSQL Left Join (First 3 cols):")
print(sql_join.tail())